# Аналитический отчет по MovieLens Dataset
## Рассказываем историю через данные о фильмах

Приветствуем в нашем исследовательском путешествии по миру кинематографа! Сегодня мы проанализируем данные MovieLens, чтобы узнать:
- Какие фильмы самые популярные?
- Как пользователи оценивают фильмы?
- Какие теги используют зрители?
- Как фильмы связаны с идентификаторами IMDb и TMDb?

### Часть 1: Импорт модуля и проверка исключений

In [1]:
import movielens_analysis as mla


Представьте: вы — исследователь, который только что получил архив с данными о тысячах фильмов и более чем ста тысячах оценок. Ваша задача — аккуратно открыть этот «цифровой сундук», не сломав замки и не потеряв ни одной ценной записи.

Но что, если файл повреждён? Или его структура не совпадает с ожидаемой?
Вот где начинается наша первая история — история про подготовку и безопасность.

Вместо того чтобы надеяться на удачу, мы пишем код, который предвидит проблемы и мягко реагирует на них.

In [2]:
%%timeit -n 1 -r 1
try:
    wrong_movies = mla.Movies("wrong_path.csv", 100)
except FileNotFoundError as e:
    print(f"Обработано исключение: {e}")
except ValueError as e:
    print(f"Обработано исключение: {e}")

try:
    wrong_ratings = mla.Ratings("wrong_ratings.csv", "tables/movies.csv", 100)
except FileNotFoundError as e:
    print(f"Обработано исключение: {e}")


Обработано исключение: File wrong_path.csv not found
Обработано исключение: File wrong_ratings.csv not found
19.9 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Часть 2: Загрузка данных и первоначальный анализ

In [3]:
movies = mla.Movies('tables/movies.csv')
ratings = mla.Ratings('tables/ratings.csv', 'tables/movies.csv')
tags = mla.Tags('tables/tags.csv')
links = mla.Links('tables/links.csv', 'tables/movies.csv')
users = mla.Users('tables/ratings.csv', 'tables/movies.csv')


Итак, данные загружены. Сундук открыт.
Но что лежит внутри? Кто наши главные герои? Сколько их? О чём они?

In [4]:
%%timeit -n 1 -r 1
movies.get_dataset_summary()


СВОДКА ПО ДАТАСЕТУ ФИЛЬМОВ
Всего фильмов: 9742
Диапазон лет: 1902 - 2018
Уникальных жанров: 19
Самый популярный жанр: Drama
Самый старый фильм: Trip to the Moon, A (Voyage dans la lune, Le) (1902)
Самый новый фильм: Avengers: Infinity War - Part I (2018)
3.68 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Ответ: Целый мир — в цифрах и названиях.

### Часть 3: Анализ фильмов по годам и жанрам

Теперь, когда мы познакомились с нашими данными, пришло время задать важные вопросы:
В какие годы снимали больше всего фильмов?
Какие жанры доминировали в разные эпохи?

Давайте отправимся в путешествие во времени и по жанровым картам кинематографа!

In [5]:
%%timeit -n 1 -r 1
all_years = movies.get_all_years()
print(f"Годы выпуска фильмов в наборе данных: от {all_years[0]} до {all_years[-1]}")


Годы выпуска фильмов в наборе данных: от 1902 до 2018
32.8 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [6]:
%%timeit -n 1 -r 1
year_distribution = movies.dist_by_release()
print(f"Всего уникальных годов: {len(year_distribution)}")
top_years = list(year_distribution.items())[:5]
print(f"Топ-5 годов по количеству фильмов: {top_years}")


Всего уникальных годов: 106
Топ-5 годов по количеству фильмов: [(2002, 311), (2006, 295), (2001, 294), (2007, 284), (2000, 283)]
2.08 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [7]:
%%timeit -n 1 -r 1
genre_stats = movies.get_genre_statistics()
print(f"Топ-5 жанров: {list(genre_stats.items())[:5]}")


Топ-5 жанров: [('Drama', 4361), ('Comedy', 3756), ('Thriller', 1894), ('Action', 1828), ('Romance', 1596)]
3.14 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [8]:
%%timeit -n 1 -r 1
multi_genre_movies = movies.most_genres(5)
print(f"Фильмы с наибольшим количеством жанров: {multi_genre_movies}")


Фильмы с наибольшим количеством жанров: {'Rubber': 10, 'Patlabor: The Movie (Kidô keisatsu patorebâ: The Movie)': 8, 'Aelita: The Queen of Mars (Aelita)': 7, 'Aqua Teen Hunger Force Colon Movie Film for Theaters': 7, 'Enchanted': 7}
3.62 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Теперь мы знаем когда и какие фильмы снимали. 
Мы переходим к самому интересному — к оценкам пользователей!
Узнаем, какие фильмы стали хитами, а какие — разочарованиями.

### Часть 4: Анализ оценок пользователей

Фильмы сняты, жанры определены, годы известны. Но самый важный вопрос оставался без ответа:
Что думают зрители?
Какие фильмы трогают сердца, а какие оставляют равнодушными?

Давайте откроем эту тайну — погрузимся в мир оценок, рейтингов и зрительских симпатий!

In [9]:
%%timeit -n 1 -r 1
rating_dist = ratings.get_rating_distribution()
total_ratings = sum(rating_dist.values())
print(f"Всего оценок: {total_ratings}")
print(f"Средняя оценка: {ratings.get_average_rating():.2f}")


Всего оценок: 100836
Средняя оценка: 3.50
10.1 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [10]:
%%timeit -n 1 -r 1
top_movies_avg = ratings.top_by_ratings(5, metric="average", min_ratings=20)
print("Топ-5 фильмов по среднему рейтингу (не менее 20 оценок):")
for i, movie in enumerate(top_movies_avg, 1):
    print(f"{i}. {movie['title']}")


Топ-5 фильмов по среднему рейтингу (не менее 20 оценок):
1. Streetcar Named Desire, A
2. Shawshank Redemption, The
3. Sunset Blvd. (a.k.a. Sunset Boulevard)
4. Philadelphia Story, The
5. In the Name of the Father
16 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [11]:
%%timeit -n 1 -r 1
top_movies_median = ratings.top_by_ratings(5, metric="median", min_ratings=20)
print("Топ-5 фильмов по медианному рейтингу (не менее 20 оценок):")
for i, movie in enumerate(top_movies_median, 1):
    print(f"{i}. {movie['title']}")


Топ-5 фильмов по медианному рейтингу (не менее 20 оценок):
1. Streetcar Named Desire, A
2. Apocalypse Now
3. Boondock Saints, The
4. Bridge on the River Kwai, The
5. Casablanca
13.4 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Продолжение следует...
В следующей части мы познакомимся с самими зрителями — теми, чьи оценки создали эту историю.

### Часть 5: Анализ тегов пользователей

Мы уже знаем, какие оценки ставят фильмам. Но оценки — это цифры. А что насчёт слов?
Что пишут люди, когда хотят выразить не просто "нравится/не нравится", а передать свои эмоции, впечатления, мысли?

Давайте заглянем в личные заметки зрителей — в мир тегов!

In [12]:
%%timeit -n 1 -r 1
tag_analysis = tags.get_tagging_analysis()
print(f"Всего тегов: {tag_analysis['total_tags']}")
print(f"Уникальных тегов: {tag_analysis['unique_tags']}")
print(f"Самый популярный тег: '{tag_analysis['most_common_tag']}'")


Всего тегов: 3683
Уникальных тегов: 1475
Самый популярный тег: 'in netflix queue'
99.2 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Уникальных тегов больше половины от общего числа! Это значит, что зрители не просто выбирают из готового списка — они творят, придумывают, ищут свои слова для описания фильмов!

In [13]:
%%timeit -n 1 -r 1
popular_tags = tags.most_popular(5)
print(f"Самые популярные теги: {popular_tags}")


Самые популярные теги: {'in netflix queue': 131, 'atmospheric': 41, 'funny': 24, 'superhero': 24, 'surreal': 24}
368 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [14]:
%%timeit -n 1 -r 1
tags_with_most_words = tags.most_words(5)
print(f"Теги с наибольшим количеством слов: {tags_with_most_words}")


Теги с наибольшим количеством слов: {'something for everyone in this one... saw it without and plan on seeing it with kids!': 16, 'the catholic church is the most corrupt organization in history': 10, 'villain nonexistent or not needed for good story': 8, '06 oscar nominated best movie - animation': 7, 'it was melodramatic and kind of dumb': 7}
920 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Человек не просто оценил фильм — он поделился личным опытом, планами, эмоциями. Это уже не метка, а история.

In [15]:
%%timeit -n 1 -r 1
longest_tags_list = tags.longest(5)
print(f"Самые длинные теги: {longest_tags_list}")


Самые длинные теги: ['something for everyone in this one... saw it without and plan on seeing it with kids!', 'the catholic church is the most corrupt organization in history', 'villain nonexistent or not needed for good story', 'r:disturbing violent content including rape', '06 oscar nominated best movie - animation']
465 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


Файл links.csv связывает внутренний идентификатор MovieLens с идентификаторами IMDb и TMDb.
Он позволяет перейти от локальной аналитики к внешним источникам, не подменяя отсутствующие данные выдуманными значениями.

### Часть 6: Внешние идентификаторы и дополнительная аналитика

In [16]:
%%timeit -n 1 -r 1
test_movies = ["Toy Story", "Jumanji", "Grumpier Old Men"]
for movie_title in test_movies:
    external_ids = links.get_external_ids(movie_title)
    imdb_url = links.get_imdb(movie_title)
    print(f"{movie_title}: {external_ids}, {imdb_url}")


Toy Story: {'imdbId': 'tt0114709', 'tmdbId': '862'}, https://www.imdb.com/title/tt0114709/
Jumanji: {'imdbId': 'tt0113497', 'tmdbId': '8844'}, https://www.imdb.com/title/tt0113497/
Grumpier Old Men: {'imdbId': 'tt0113228', 'tmdbId': '15602'}, https://www.imdb.com/title/tt0113228/
116 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [17]:
%%timeit -n 1 -r 1
print(f"Фильмов с внешними идентификаторами: {len(links.joined_data)}")


Фильмов с внешними идентификаторами: 9742
23.5 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [18]:
%%timeit -n 1 -r 1
ratings_by_year = ratings.dist_by_year()
print(f"Распределение оценок по годам: {ratings_by_year}")


Распределение оценок по годам: {1996: 6040, 1997: 1916, 1998: 507, 1999: 2439, 2000: 10061, 2001: 3922, 2002: 3478, 2003: 4014, 2004: 3279, 2005: 5813, 2006: 4059, 2007: 7114, 2008: 4351, 2009: 4158, 2010: 2301, 2011: 1690, 2012: 4656, 2013: 1664, 2014: 1439, 2015: 6616, 2016: 6703, 2017: 8198, 2018: 6418}
6.97 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [19]:
%%timeit -n 1 -r 1
most_rated = ratings.top_by_num_of_ratings(5)
print(f"Фильмы с наибольшим числом оценок: {most_rated}")


Фильмы с наибольшим числом оценок: {'Forrest Gump': 329, 'Shawshank Redemption, The': 317, 'Pulp Fiction': 307, 'Silence of the Lambs, The': 279, 'Matrix, The': 278}
5.05 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [20]:
%%timeit -n 1 -r 1
controversial = ratings.top_controversial(5, min_ratings=20)
print(f"Фильмы с наибольшей дисперсией оценок: {controversial}")


Фильмы с наибольшей дисперсией оценок: {'Thin Red Line, The': 1.9457142857142855, 'Blair Witch Project, The': 1.857177734375, 'Barb Wire': 1.8525000000000003, 'Home Alone 2: Lost in New York': 1.798126951092612, 'Girl with the Dragon Tattoo, The': 1.7200963718820863}
11.3 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [21]:
%%timeit -n 1 -r 1
active_users = sorted(
    ((user_id, len(values)) for user_id, values in users.user_ratings.items()),
    key=lambda item: (-item[1], item[0]),
)[:5]
print(f"Самые активные пользователи: {active_users}")


Самые активные пользователи: [(414, 2698), (599, 2478), (474, 2108), (448, 1864), (274, 1346)]
523 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [22]:
%%timeit -n 1 -r 1
user_variance = users.top_n_by_ratings_variance(5)
print(f"Пользователи с наибольшей дисперсией оценок: {user_variance}")


Пользователи с наибольшей дисперсией оценок: [(3, 4.25871137409599, 39), (461, 3.098765432098765, 27), (55, 3.0944, 25), (259, 2.942330558858502, 29), (329, 2.9177693761814747, 23)]
6.19 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Часть 7: Дополнительные анализы

Мы многое узнали о фильмах в целом. Но что если вам нужен конкретный фильм?
Или вы хотите посмотреть все фильмы 1995 года? Или все драмы?
Или найти самый старый фильм в коллекции?

Давайте превратимся в кино-детективов и научимся находить любые фильмы за секунды!

In [23]:
%%timeit -n 1 -r 1
movie = movies.get_movie_by_title("Toy Story")
if movie:
    print(f"Найден фильм: {movie['title']} ({movie['year']})")
    print(f"Жанры: {', '.join(movie['genres'])}")


Найден фильм: Toy Story (1995)
Жанры: Adventure, Animation, Children, Comedy, Fantasy
29.7 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [24]:
%%timeit -n 1 -r 1
movies_1995 = movies.get_movies_by_year(1995)
print(f"Фильмов 1995 года: {len(movies_1995)}")
if movies_1995:
    print(f"Пример: {movies_1995[0]['title']}")


Фильмов 1995 года: 259
Пример: 3 Ninjas Knuckle Up
84.1 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [25]:
%%timeit -n 1 -r 1
drama_movies = movies.get_movies_by_genre("Drama")
print(f"Фильмов в жанре Drama: {len(drama_movies)}")


Фильмов в жанре Drama: 4361
974 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [26]:
%%timeit -n 1 -r 1
oldest, newest = movies.get_oldest_newest_movies()
if oldest and newest:
    print(f"Самый старый фильм: {oldest['title']} ({oldest['year']})")
    print(f"Самый новый фильм: {newest['title']} ({newest['year']})")


Самый старый фильм: Trip to the Moon, A (Voyage dans la lune, Le) (1902)
Самый новый фильм: Avengers: Infinity War - Part I (2018)
953 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [27]:
%%timeit -n 1 -r 1
year_analysis = movies.get_year_analysis(1995)
print(f"Анализ 1995 года: {year_analysis}")


Анализ 1995 года: {'total_movies': 259, 'animation_count': 13, 'animation_percentage': 5.019305019305019, 'is_special_year': True}
112 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [28]:
%%timeit -n 1 -r 1
genre_dist = movies.dist_by_genres()
print(f"Распределение по жанрам (первые 5): {list(genre_dist.items())[:5]}")


Распределение по жанрам (первые 5): [('Drama', 4361), ('Comedy', 3756), ('Thriller', 1894), ('Action', 1828), ('Romance', 1596)]
2.54 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [29]:
%%timeit -n 1 -r 1
if ratings.user_ratings:
    user_id = list(ratings.user_ratings.keys())[0]
    user_ratings = ratings.get_user_ratings(user_id)
    print(f"Пользователь {user_id} поставил {len(user_ratings)} оценок")
    if user_ratings:
        movie_info = movies.get_movie_by_id(user_ratings[0]['movieId'])
        if movie_info:
            print(f"Высшая оценка: {movie_info['title']} - {user_ratings[0]['rating']}")


Пользователь 1 поставил 232 оценок
Высшая оценка: Seven (a.k.a. Se7en) - 5.0
65 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [30]:
%%timeit -n 1 -r 1
if ratings.movie_ratings:
    movie_id = list(ratings.movie_ratings.keys())[0]
    avg_rating = ratings.get_average_rating_for_movie(movie_id)
    movie_info = movies.get_movie_by_id(movie_id)
    if movie_info:
        print(f"Средний рейтинг для '{movie_info['title']}': {avg_rating:.2f}")


Средний рейтинг для 'Toy Story': 3.92
221 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [31]:
%%timeit -n 1 -r 1
top_ids = ratings.get_top_movies_ids(3, min_ratings=20)
print(f"ID и названия топ-3 фильмов: {top_ids}")


ID и названия топ-3 фильмов: [(1104, 'Streetcar Named Desire, A'), (318, 'Shawshank Redemption, The'), (922, 'Sunset Blvd. (a.k.a. Sunset Boulevard)')]
12.1 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [32]:
%%timeit -n 1 -r 1
if tags.movie_tags:
    movie_id = list(tags.movie_tags.keys())[0]
    movie_tags = tags.get_tags_for_movie(movie_id)
    movie_info = movies.get_movie_by_id(movie_id)
    if movie_info:
        print(f"Теги для фильма '{movie_info['title']}': {movie_tags[:5]}")


Теги для фильма 'Step Brothers': ['comedy', 'funny', 'highly quotable', 'will ferrell']
43 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [33]:
%%timeit -n 1 -r 1
if tags.user_tags:
    user_id = list(tags.user_tags.keys())[0]
    user_tags = tags.get_tags_by_user(user_id)
    print(f"Пользователь {user_id} использовал {len(user_tags)} уникальных тегов")


Пользователь 2 использовал 9 уникальных тегов
25 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [34]:
%%timeit -n 1 -r 1
intersection = tags.most_words_and_longest(5)
print(f"Теги, которые одновременно длинные и содержат много слов: {intersection}")


Теги, которые одновременно длинные и содержат много слов: ['06 oscar nominated best movie - animation', 'something for everyone in this one... saw it without and plan on seeing it with kids!', 'the catholic church is the most corrupt organization in history', 'villain nonexistent or not needed for good story']
2.29 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [35]:
%%timeit -n 1 -r 1
action_tags = tags.tags_with("action")
print(f"Теги, содержащие слово 'action': {len(action_tags)} штук")


Теги, содержащие слово 'action': 6 штук
307 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


### Часть 8: Заключение и выводы

## Выводы

### Что мы узнали из анализа MovieLens?

1. **Объём данных**: Проанализированы 9742 фильма, 100836 оценок и 3683 тега
2. **Популярность жанров**: Драма — самый частый жанр, он встречается у 4361 фильма
3. **Оценки пользователей**: Средняя оценка составляет 3.50, а чаще всего пользователи ставят 4.0
4. **Популярность фильмов**: Больше всего оценок получил фильм «Forrest Gump» — 329
5. **Хронология**: Данные охватывают фильмы с 1902 по 2018 год

### Интересные находки:
- Самая частая оценка — 4.0
- Самый популярный тег — `in netflix queue`
- У некоторых фильмов указано до 10 жанров
- Внешние ID доступны для всех 9742 фильмов